# Clinical MCQ — LoRA fine-tune (Phi-3.5-mini) — overnight, full-data run

The earlier run trained on **30k-dataset + medium-shuffled** (27,157 examples after
dedup) and finished in ~2.2h — but its own loss curve showed validation loss had
essentially flattened after the first ~100 of 1,698 steps. That means the model had
already extracted most of what that data had to teach it; more *epochs* on the same
27k questions would mostly be re-showing it things it's already fit, which risks
memorization more than learning.

So this run changes the actual lever that matters: **more unique data**, not more
passes over the same data. It merges every training source available — the two from
before, plus the competition's own `train_*.jsonl` files (`train_easy`, `train_medium`,
and `train_hard` if it exists) — deduplicated across all of them. Still **1 epoch**,
same LoRA capacity (r=16) as before, so the only thing that changed between the two
runs is data volume — a clean before/after to compare tomorrow.

Saves `loramodel_final` for the RAG/eval notebook to score, same as before.

Every non-obvious setting below is chosen from something measured on this exact
Kaggle T4, not from defaults:

| Setting | Value | Why |
|---|---|---|
| `per_device_batch_size` | auto (probe picks it) | Benchmarked 2.84 / 3.04 / 3.08 ex/s at batch 4 / 8 / 16 on the previous run, then a **regression** to 2.91 at 32 — the GPU was saturated, 16 was the ceiling then. Re-probed here since a bigger dataset changes the worst-case sequence-length batch. |
| `gradient_accumulation` | derived | Keeps effective batch at 16 regardless of what the probe picks. |
| gradient checkpointing | **off** | Peak memory measured at 3.15 GB of 15 GB last run — the memory it protects was never scarce. Off = ~25% less compute with *identical* gradients. |
| `max_seq_length` | 1024, no truncation | Unsloth auto-enables padding-free, so a wide window costs nothing; lowering it would only throw away the long tail. |
| eval subset | 1,000 rows | Evaluating the full split repeatedly once cost ~19h — **more than training itself**. Eval never updates weights, so a subset gives the same curve. |
| `lr_scheduler` | cosine | Requested. |
| `MAX_TRAIN_EXAMPLES` | overnight budget guard | This is an **unattended** run — a session-cap timeout at hour 11 with nothing planned for it is worse than capping deliberately. Section 6 projects wall-clock from measured throughput and caps if it's over budget, rather than finding out at 6am. |

Also included, because they decide whether the final numbers mean anything:
**cross-source dedup** (these sources overlap a lot — the last run found 30k-dataset
already contained ~98% of medium-shuffled) and a **contamination check** against the
competition eval/test sets.

> Settings → Accelerator **GPU T4**, Internet **On**. Checkpoints every 250 steps —
> if the session disconnects overnight, re-run from section 10 and it resumes rather
> than restarting.

In [ ]:
%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    # Pins verified against live PyPI metadata + source, not guessed:
    #  - unsloth requires trl!=0.19.0,<=0.24.0,>=0.18.2 and peft!=0.11.0,>=0.18.0
    #  - trl 0.24.0 requires transformers>=4.56.1
    #  - peft's utils/constants.py does an unguarded
    #    `from transformers import BloomPreTrainedModel`, which breaks on
    #    transformers>=5.0 where that name left the top-level lazy-import table.
    # => transformers must sit in [4.56.1, 5.0). Installing the family together
    #    lets pip solve all constraints at once instead of one at a time.
    import subprocess
    subprocess.run(
        ["pip", "install", "-q", "-U",
         "unsloth", "unsloth_zoo",
         "transformers>=4.56.1,<5.0",
         "trl>=0.18.2,<=0.24.0,!=0.19.0",
         "peft>=0.18.0,!=0.11.0",
         "accelerate>=0.34.1"],
        check=True,
    )
else:
    !pip install --no-deps bitsandbytes "accelerate>=0.34.1" xformers==0.0.29.post3 "peft>=0.18.0,!=0.11.0" "trl>=0.18.2,<=0.24.0,!=0.19.0" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" huggingface_hub hf_transfer "transformers>=4.56.1,<5.0"
    !pip install --no-deps unsloth

In [ ]:
import transformers, trl, peft, torch
print("transformers:", transformers.__version__)
print("trl         :", trl.__version__)
print("peft        :", peft.__version__)
print("CUDA        :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Config

In [ ]:
import os, json, re, random, glob as _glob
import numpy as np
import torch

SEED = 3407
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_MODEL     = "unsloth/Phi-3.5-mini-instruct"
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT   = True

COMP = "/kaggle/input/goedel-machines-x-iitm-clinical-llm-challenge"

# Found by filename, not assumed folder slug -- a hardcoded "/kaggle/input/30k-dataset/..."
# path has silently missed an attached dataset before (Kaggle's mount folder name isn't
# always the display name). Merges everything available: the two sources from the
# earlier run, plus whatever train_*.jsonl the competition itself ships (easy/medium/
# hard, whichever exist) -- more unique questions than any single source alone.
def _find_by_filename(filename, base="/kaggle/input"):
    hits = sorted(_glob.glob(f"{base}/**/{filename}", recursive=True))
    return hits[0] if hits else None

TRAIN_SOURCES = [p for p in [
    _find_by_filename("randomized_output.jsonl"),
    _find_by_filename("shuffled_input_medum.jsonl"),
] if p] + sorted(_glob.glob(f"{COMP}/train_*.jsonl"))
TRAIN_SOURCES = sorted(set(TRAIN_SOURCES))

# Held-out sets, used only for the contamination check (never trained on).
EVAL_FILES = [f"{COMP}/sample_test_medium.jsonl", f"{COMP}/sample_test_hard.jsonl",
              f"{COMP}/test_easy.jsonl", f"{COMP}/test_medium.jsonl"]

ADAPTER_OUT = "loramodel_final"   # the eval notebook's LORA_PATH must match this
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Train sources found:")
for p in TRAIN_SOURCES:
    print(f"  ✅ {p}")
assert TRAIN_SOURCES, (
    "No training files found. Attach '30k-dataset', 'mediam shufled', and/or the "
    "competition dataset (for train_easy.jsonl / train_medium.jsonl)."
)

## 2. Load and inspect schemas

`randomized_output.jsonl` was checked offline before writing this: 30,000 rows, every
one `{id, question, options{A–D}, answer}`, zero malformed, 4 options on every row, and
only 4 internal duplicate questions. The competition's own `train_*.jsonl` files
haven't been checked the same way — schemas are printed for every source below, and
the normaliser in the next section handles common variations rather than assuming
they all match. A schema drift on re-upload should surface loudly here, not silently
shrink or corrupt the training set.

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"  ⚠️  malformed line {i} in {os.path.basename(path)}")
    return rows

raw_sources = {}
for path in TRAIN_SOURCES:
    rows = load_jsonl(path)
    raw_sources[path] = rows
    print(f"\n=== {os.path.basename(path)} : {len(rows):,} rows ===")
    print("fields:", sorted(rows[0].keys()))
    print(json.dumps(rows[0], indent=2)[:900])

## 3. Normalise to one schema

Handles the common shapes: options as a dict (`{"A": ...}`) or a list, and the answer
given as a letter, an index, or the full option text. Rows that can't be normalised
are counted and reported — a silent drop here would quietly shrink the training set.

In [ ]:
LETTERS = "ABCD"

def normalise(row):
    """-> {question, options{A..D}, answer} or None if the row can't be parsed."""
    q = row.get("question") or row.get("Question") or row.get("instruction")
    if not q or not str(q).strip():
        return None

    opts = row.get("options", row.get("Options", row.get("choices")))
    if isinstance(opts, dict):
        options = {str(k).strip().upper()[:1]: str(v).strip() for k, v in opts.items()}
    elif isinstance(opts, (list, tuple)):
        options = {LETTERS[i]: str(v).strip() for i, v in enumerate(opts) if i < 4}
    else:
        return None
    if not all(L in options and options[L] for L in LETTERS):
        return None

    ans = row.get("answer", row.get("Answer", row.get("label")))
    if ans is None:
        return None
    a = str(ans).strip()
    if a.upper()[:1] in LETTERS and len(a) <= 2:          # "A" / "A."
        letter = a.upper()[:1]
    elif a.isdigit() and int(a) < 4:                       # index
        letter = LETTERS[int(a)]
    else:                                                  # full option text
        match = [L for L, v in options.items() if v.lower() == a.lower()]
        if not match:
            return None
        letter = match[0]
    return {"question": str(q).strip(), "options": options, "answer": letter}


normalised, per_source = [], {}
for path, rows in raw_sources.items():
    good = [n for n in (normalise(r) for r in rows) if n]
    per_source[path] = len(good)
    normalised += good
    dropped = len(rows) - len(good)
    flag = "  ⚠️  investigate" if dropped > len(rows) * 0.02 else ""
    print(f"{os.path.basename(path):32s} {len(good):>7,} kept  {dropped:>6,} dropped{flag}")

print(f"\nTOTAL normalised: {len(normalised):,}")
assert normalised, "Nothing normalised -- check the schema printout in section 2."

## 4. Dedup + contamination check

Two different questions worth answering before training:

1. **Do the two sources overlap?** Duplicates would over-weight those examples.
2. **Do any training questions appear in the eval/test sets?** If so the accuracy
   later is inflated, and that has to be reported rather than discovered by a reader.

In [ ]:
def norm_q(s):
    return re.sub(r"\s+", " ", str(s)).strip().lower()

seen, deduped, dupes = set(), [], 0
for ex in normalised:
    k = norm_q(ex["question"])
    if k in seen:
        dupes += 1
        continue
    seen.add(k)
    deduped.append(ex)

print(f"cross-source dedup: {len(normalised):,} -> {len(deduped):,}  ({dupes:,} duplicates removed)")

train_qs = {norm_q(e["question"]) for e in deduped}
contamination = {}
print()
for path in EVAL_FILES:
    if not os.path.exists(path):
        print(f"  (missing, skipped) {os.path.basename(path)}")
        continue
    ev = {norm_q(r["question"]) for r in load_jsonl(path) if "question" in r}
    n = len(ev & train_qs)
    pct = 100 * n / max(len(ev), 1)
    contamination[os.path.basename(path)] = {"n_eval": len(ev), "overlap": n, "pct": round(pct, 2)}
    flag = "  ⚠️  INFLATES RESULTS -- report this" if pct > 1 else ""
    print(f"  {os.path.basename(path):28s} {n:>6,}/{len(ev):<6,} overlap = {pct:5.2f}%{flag}")

with open(f"{RESULTS_DIR}/contamination.json", "w") as f:
    json.dump(contamination, f, indent=2)
print(f"\nsaved -> {RESULTS_DIR}/contamination.json")

# Answer-position balance. The 30k source measured 24.8/24.9/25.1/25.3 offline --
# its options were deliberately randomised, so there is no positional shortcut to
# learn. If the MERGED set were skewed, a model could score above chance by just
# favouring one letter, which would make the eval accuracy meaningless.
from collections import Counter
dist = Counter(e["answer"] for e in deduped)
top = max(dist.values()) / len(deduped)
print("\nanswer distribution of merged training set:")
for L in LETTERS:
    print(f"  {L}: {dist[L]:>7,}  {100*dist[L]/len(deduped):5.2f}%")
print("  ✅ balanced -- no positional shortcut" if top < 0.30 else
      f"  ⚠️  SKEWED ({100*top:.1f}% on one letter) -- a constant-guess baseline would beat chance")

## 5. Prompt template

**This must match the eval notebook byte-for-byte up to `<|assistant|>`.** A mismatch
here is silent and expensive: the adapter gets queried in a format it never saw, and
the fine-tune looks like it did nothing. The constants below are the same ones the
eval notebook builds its prompt from — keep them in sync.

The target begins with the bare option letter, which is what makes logit-based
scoring at eval time line up with what was actually trained.

In [ ]:
SYSTEM_MSG = ("You are preparing medical students for board examinations (USMLE, COMLEX). "
              "Provide accurate, evidence-based answers consistent with current medical standards.")
USER_HEAD  = ("This is a medical board examination question. Apply your knowledge of clinical "
              "medicine, basic sciences, and current guidelines.")
USER_TAIL  = ("Choose the letter corresponding to the BEST answer based on current medical "
              "knowledge and clinical practice guidelines.")

def format_options(options):
    return "\n".join(f"{k}. {options[k]}" for k in LETTERS if k in options)

def build_prompt(question, options, context=None):
    """Identical to the eval notebook's build_prompt -- returns text up to <|assistant|>."""
    body = USER_HEAD + "\n\n"
    if context:
        body += f"### Context:\n{context}\n\n"
    body += f"### Question:\n{question.strip()}\n\n"
    body += f"### Options:\n{format_options(options)}\n\n"
    body += USER_TAIL
    return (f"<|system|>\n{SYSTEM_MSG}<|end|>\n"
            f"<|user|>\n{body}<|end|>\n"
            f"<|assistant|>\n")

def to_training_text(ex):
    letter = ex["answer"]
    return {"text": build_prompt(ex["question"], ex["options"])
                    + f"{letter}. {ex['options'][letter]}<|end|>"}

print(to_training_text(deduped[0])["text"])

## 6. Build the dataset

The split happens on the **raw** records first, so the held-out portion keeps its
`question` / `options` / `answer` fields and can be written out as a real scoreable
eval set.

That matters more than it looks. The competition's own dev files are tiny —
`sample_test_medium` and `sample_test_hard` are **10 rows each**, `test_easy` and
`test_medium` 100 each. At n=10 a genuinely 75%-accurate model carries a ±27
percentage-point confidence interval, so those files physically cannot tell a good
model from a bad one. The ~3k held-out split written here gives ±1.5pp instead, which
is what a base-vs-fine-tuned comparison actually needs.

In [ ]:
import random as _rnd
from datasets import Dataset

rows = list(deduped)
_rnd.Random(SEED).shuffle(rows)

n_val = max(1, int(len(rows) * 0.1))
val_raw, train_raw = rows[:n_val], rows[n_val:]

# Overnight budget guard. This run is unattended, so a session-cap timeout at
# hour 11 with nothing planned for it is a worse outcome than capping on
# purpose now. Projected from the previous run's measured throughput
# (~3.3-3.4 ex/s with checkpointing off, on this same T4). train_raw is
# already globally shuffled above, so truncating it keeps a representative mix
# rather than cutting off whichever source happened to sort last.
MEASURED_EX_PER_SEC = 3.4
TARGET_HOURS = 10          # "something like 10h overnight"
MAX_TRAIN_EXAMPLES = None  # set an int to force a cap; None = use the projection below

projected_all_h = len(train_raw) / MEASURED_EX_PER_SEC / 3600
print(f"available training examples: {len(train_raw):,}  (~{projected_all_h:.1f} h uncapped)")

if MAX_TRAIN_EXAMPLES is None and projected_all_h > TARGET_HOURS * 1.1:
    MAX_TRAIN_EXAMPLES = int(TARGET_HOURS * 3600 * MEASURED_EX_PER_SEC)
    print(f"⏱️  capping to fit ~{TARGET_HOURS}h: {len(train_raw):,} -> {MAX_TRAIN_EXAMPLES:,} examples")

if MAX_TRAIN_EXAMPLES and len(train_raw) > MAX_TRAIN_EXAMPLES:
    train_raw = train_raw[:MAX_TRAIN_EXAMPLES]

projected_h = len(train_raw) / MEASURED_EX_PER_SEC / 3600
print(f"\n⏱️  training on {len(train_raw):,} examples at ~{MEASURED_EX_PER_SEC} ex/s -> ~{projected_h:.1f} h")
if projected_h > 11:
    print("   ⚠️  Still near/over Kaggle's 12h session cap. Lower MAX_TRAIN_EXAMPLES above and re-run this cell.")
else:
    print("   ✅ fits an overnight session with margin. (Estimate only -- the trainer")
    print("      cell's probe measures real throughput; if it differs a lot, adjust")
    print("      MAX_TRAIN_EXAMPLES here and re-run from this cell.)")

# Held-out set WITH the original fields, for the eval notebook to score on.
HELD_OUT = f"{RESULTS_DIR}/held_out_val.jsonl"
with open(HELD_OUT, "w") as f:
    for r in val_raw:
        f.write(json.dumps(r) + "\n")

train_dataset = Dataset.from_list([to_training_text(e) for e in train_raw])

# Mid-training eval only draws the loss curve -- it never updates weights, so its
# size has zero effect on the trained model. A subset keeps it cheap.
EVAL_SUBSET = 1000
val_dataset = Dataset.from_list([to_training_text(e) for e in val_raw[:EVAL_SUBSET]])

print(f"\ntrain                  : {len(train_raw):,}")
print(f"held-out val (scored)  : {len(val_raw):,}  -> {HELD_OUT}")
print(f"val used for loss curve: {len(val_dataset):,}")
assert len(train_raw) > len(val_raw)
print(f"\n➡️  In the eval notebook, score on {HELD_OUT} (~{len(val_raw):,} rows,")
print( "    never trained on) rather than the 10-row competition dev files.")

## 7. Model + LoRA

In [ ]:
from unsloth import FastLanguageModel

# Off, on measured evidence: peak memory was 3.15 GB of 15 GB and p90 sequence
# length is 223 tokens, so the memory checkpointing protects was never scarce.
# Gradients are identical either way -- this is purely time-vs-memory, and time
# is the binding constraint. Set to "unsloth" if the memory probe below warns.
USE_GRADIENT_CHECKPOINTING = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = USE_GRADIENT_CHECKPOINTING,
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")
print(f"gradient checkpointing = {USE_GRADIENT_CHECKPOINTING}")
assert trainable > 0, "LoRA did not attach."

## 8. Memory safety probe

With checkpointing off, activation memory scales with `batch × seq_len`. This runs one
deliberately pessimistic step — a full batch of the longest example in the data — so an
OOM shows up in ~30 seconds rather than in hour 2 of the run.

In [ ]:
import gc

lengths = np.array([
    len(tokenizer(train_dataset[i]["text"], add_special_tokens=False)["input_ids"])
    for i in range(min(1000, len(train_dataset)))
])
p50, p90, p99 = np.percentile(lengths, [50, 90, 99])
print(f"token lengths: mean={lengths.mean():.0f} p50={p50:.0f} p90={p90:.0f} "
      f"p99={p99:.0f} max={lengths.max()}  (cap {MAX_SEQ_LENGTH})")

worst = int(min(MAX_SEQ_LENGTH, lengths.max()))
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9


def probe(bs, seqlen):
    """Peak GB for one fwd+bwd, or None on OOM."""
    torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
    try:
        ids = torch.randint(100, 30000, (bs, seqlen), device="cuda")
        model.train()
        model(input_ids=ids, labels=ids).loss.backward()
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / 1e9
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        return None
    finally:
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache(); gc.collect()


# Search down for the largest batch that survives the worst case, instead of
# failing on one guess. Only batches that divide 16 are considered, so
# grad_accum can hold the effective batch at 16 and the training maths is
# identical whichever one wins.
#
# NOTE this probe is deliberately harsh: a full batch of the single longest
# example (~721 tokens) is ~3x the typical batch (~232 tokens each), and since
# group_by_length is off, batches are randomly composed -- so a real batch that
# extreme essentially never occurs. Passing here means training is safe with
# margin, not merely likely to survive.
print(f"\nworst-case probe: batch x {worst} tokens, {total_gb:.1f} GB card")
PER_DEVICE_BS = None
for bs in (16, 8, 4, 2):
    peak = probe(bs, worst)
    if peak is None:
        print(f"   batch {bs:>2}: OOM")
        continue
    head = total_gb - peak
    print(f"   batch {bs:>2}: peak {peak:5.2f} GB, {head:4.1f} GB headroom"
          f"{'  <- selected' if head > 2.0 and PER_DEVICE_BS is None else ''}")
    if head > 2.0 and PER_DEVICE_BS is None:
        PER_DEVICE_BS = bs
        break

if PER_DEVICE_BS is None:
    PER_DEVICE_BS = 4
    print("\n⚠️  No batch cleared 2 GB headroom with checkpointing off.")
    print("   Set USE_GRADIENT_CHECKPOINTING='unsloth' in section 7 and re-run from there.")
    print("   That costs ~25% speed but makes activation memory nearly flat.")
else:
    print(f"\n✅ PER_DEVICE_BS = {PER_DEVICE_BS} "
          f"(grad_accum {16 // PER_DEVICE_BS}, effective batch stays 16)")

## 9. Trainer

Argument placement is resolved by inspecting the installed `trl` signatures rather than
hardcoding one version's names — `dataset_text_field` / `max_seq_length` / `packing`
have moved between `SFTConfig` and `SFTTrainer` across releases, and `tokenizer=` was
renamed `processing_class=`. Anything this version doesn't recognise is dropped with a
warning instead of raising.

In [ ]:
import inspect
from trl import SFTConfig, SFTTrainer

EFFECTIVE_BATCH = 16
GRAD_ACCUM = max(1, EFFECTIVE_BATCH // PER_DEVICE_BS)
assert PER_DEVICE_BS * GRAD_ACCUM == EFFECTIVE_BATCH, (
    "PER_DEVICE_BS must divide EFFECTIVE_BATCH so the optimisation maths is unchanged."
)

bf16_ok = torch.cuda.is_bf16_supported()   # False on T4 (Turing) -> fp16

requested = dict(
    per_device_train_batch_size = PER_DEVICE_BS,
    per_device_eval_batch_size  = 16,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs   = 1,
    learning_rate      = 2e-4,
    lr_scheduler_type  = "cosine",
    warmup_ratio       = 0.03,
    weight_decay       = 0.01,
    optim              = "adamw_torch",
    neftune_noise_alpha= 5,
    logging_steps      = 10,
    eval_strategy      = "steps",
    eval_steps         = 250,
    save_strategy      = "steps",
    save_steps         = 250,
    save_total_limit   = 2,
    fp16 = not bf16_ok,
    bf16 = bf16_ok,
    dataloader_num_workers = 2,
    seed = SEED,
    output_dir = "outputs",
    report_to = "none",
)
movable = {
    "dataset_text_field": "text",
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_length": MAX_SEQ_LENGTH,
    "packing": False,
    "dataset_num_proc": min(8, os.cpu_count() or 1),
}

cfg_params = inspect.signature(SFTConfig.__init__).parameters
trn_params = inspect.signature(SFTTrainer.__init__).parameters

cfg_kwargs = {k: v for k, v in requested.items() if k in cfg_params}
if (dropped := [k for k in requested if k not in cfg_params]):
    print(f"⚠️  SFTConfig doesn't accept {dropped} in this trl version -- using its defaults.")
for k, v in movable.items():
    if k in cfg_params:
        cfg_kwargs.setdefault(k, v)

trainer_kwargs = dict(model=model, train_dataset=train_dataset,
                      eval_dataset=val_dataset, args=SFTConfig(**cfg_kwargs))
if "processing_class" in trn_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trn_params:
    trainer_kwargs["tokenizer"] = tokenizer
for k, v in movable.items():
    if k in trn_params and k not in cfg_kwargs:
        trainer_kwargs[k] = v

trainer = SFTTrainer(**trainer_kwargs)

steps = len(train_dataset) // EFFECTIVE_BATCH
print(f"\n✅ trainer ready — {len(train_dataset):,} examples, ~{steps:,} steps, "
      f"effective batch {EFFECTIVE_BATCH}, {'bf16' if bf16_ok else 'fp16'}")
print(f"   at ~3.4 ex/s (measured, checkpointing off) -> ~{len(train_dataset)/3.4/3600:.1f} h")

## 10. Train

In [ ]:
import glob

# Resume if a checkpoint exists (e.g. after a disconnect) rather than restarting.
ckpts = sorted(glob.glob("outputs/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
resume = ckpts[-1] if ckpts else None
print(f"resuming from {resume}" if resume else "starting fresh")

trainer_stats = trainer.train(resume_from_checkpoint=resume)

## 11. Save

In [ ]:
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)

with open(f"{RESULTS_DIR}/train_log_history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

print(f"✅ adapter -> {ADAPTER_OUT}")
print(f"✅ loss history -> {RESULTS_DIR}/train_log_history.json")
print(f"\nIn the eval notebook set:  LORA_PATH_LOCAL = "
      f'"/kaggle/input/<this-notebook-slug>/{ADAPTER_OUT}"')

## 12. Loss curve

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in history if "loss" in h and "step" in h]
ev = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h and "step" in h]

fig, ax = plt.subplots(figsize=(9, 5))
if tr:
    ax.plot(*zip(*tr), lw=1, alpha=.4, label="train")
    w = max(1, len(tr) // 40)
    if w > 1:
        xs, ys = zip(*tr)
        ax.plot(xs, [sum(ys[max(0, i-w):i+1]) / len(ys[max(0, i-w):i+1])
                     for i in range(len(ys))], lw=2, label=f"train (smoothed)")
if ev:
    ax.plot(*zip(*ev), "o-", lw=2, ms=5, label="validation")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=.3)
ax.set_title("LoRA fine-tune loss")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/loss_curve.png", dpi=150); plt.show()

# Did the back half of training earn its wall-clock? Worth knowing, and worth
# reporting -- "measured where returns diminished" beats "trained on everything".
if len(ev) >= 4:
    xs, ys = zip(*ev)
    mid = len(ys) // 2
    first, second = ys[0] - ys[mid], ys[mid] - ys[-1]
    best = min(range(len(ys)), key=lambda i: ys[i])
    print(f"val loss {ys[0]:.4f} (step {xs[0]}) -> {ys[-1]:.4f} (step {xs[-1]})")
    print(f"  best {ys[best]:.4f} @ step {xs[best]}"
          + ("   <-- best was NOT the final step" if best < len(ys) - 1 else ""))
    print(f"  first half {first:+.4f} | second half {second:+.4f}")
    print("  -> diminishing returns; less data would have done nearly as well."
          if second < first * 0.25 else
          "  -> still improving at the end; the data volume was justified.")

## 13. Push to Hub (optional)

Not required — the eval notebook reads the local `loramodel_final` folder via this
notebook's output. Skipped cleanly when no token is set, so it can't fail a Run All.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    print(f"ℹ️  no HF_TOKEN — skipping push. '{ADAPTER_OUT}' is saved locally, "
          "which is all the eval notebook needs.")
else:
    HF_REPO = "mouryesh/phi35-clinical-lora"
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)   # same repo as the adapter
    print(f"✅ pushed to {HF_REPO}")